# Baseline Modeling – AI4I Predictive Maintenance

This notebook develops baseline classification models to predict machine failure.
Due to class imbalance, evaluation focuses on recall and F1-score rather than accuracy.

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/ai4i2020.csv")

predictor_features = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

X = df[predictor_features]
y = df["Machine failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Baseline Model – Logistic Regression

Logistic Regression predicts the probability of machine failure (0 or 1). Instead of predicting a raw number, it outputs a probability between 0 and 1 using a sigmoid function.

This was chosen as the first model because:
- It is simple, fast, and interpretable.
- It handles class imbalance with weighting.
- It gives probability scores, not just hard predictions.
- It is interpretable and easy to evaluate.

## Feature Scaling

Logistic Regression is distance-based and sensitive to scale. Standardizing the features ensures each variable contributes fairly and prevents larger-magnitude features from overpowering smaller ones.

In [21]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Logistic Regression serves as the baseline model. Since failures are rare, `class_weight="balanced"` ensures the model does not ignore the minority class.

In [22]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight="balanced", random_state=42)
model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


## Model Predictions

- `y_pred` gives the model’s actual 0/1 decisions.
- `y_prob` gives the confidence score for failure.
- Probabilities allow adjusting the decision threshold instead of relying only on default 0.5.

In [23]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

The model is evaluated using recall, precision, F1-score, and ROC-AUC. Since the dataset is imbalanced, recall for the failure class is prioritized over overall accuracy, since missing failures is more costly than generating false alarms.

In [24]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_prob))

Confusion Matrix:
[[1585  347]
 [  11   57]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.82      0.90      1932
           1       0.14      0.84      0.24        68

    accuracy                           0.82      2000
   macro avg       0.57      0.83      0.57      2000
weighted avg       0.96      0.82      0.88      2000


ROC-AUC Score:
0.907974059188893


## Confusion Matrix Interpretation

1585 → True Negatives  
347 → False Positives  
11 → False Negatives  
57 → True Positives 

The model correctly identified 57 out of 68 actual failures (recall = 0.84), missing only 11 failures.  

However, 347 machines were incorrectly flagged as failures, leading to low precision (0.14) for the failure class.

Overall accuracy is 0.82, but this is influenced by class imbalance and is not the primary evaluation metric.

The ROC-AUC score of 0.91 indicates strong overall class separation ability.

## Model Evaluation Summary

- Accuracy alone is misleading due to class imbalance.
- Recall for failures is prioritized since missed failures are more costly.
- The balanced class weight helps the model focus on rare failure cases.
- The model achieves high recall (0.84) but at the cost of many false positives (347), showing a precision–recall tradeoff.
- This serves as a baseline for comparison with more complex models.